# Interactive Kalmanorix Demo

This notebook demonstrates the core concepts of the Kalmanorix framework:

1. **Semantic Routing with Dynamic Thresholding** - Intelligent selection of specialist models based on query similarity
2. **Kalman Fusion** - Precision-weighted fusion of embeddings using uncertainty estimates
3. **Visualization** - Real-time exploration of routing decisions and fusion weights

## Setup

First, install required packages if not already available:


In [ ]:
# Install Kalmanorix in development mode (if running locally)
# !pip install -e .
# Install optional dependencies for visualization
# !pip install matplotlib ipywidgets

import sys

sys.path.insert(0, "..")  # Add parent directory to path for development

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from typing import Set

from kalmanorix import (
    SEF,
    Village,
    ScoutRouter,
    Panoramix,
    MeanFuser,
    KalmanorixFuser,
    LearnedGateFuser,
    threshold_top_k,
    threshold_relative_to_max,
    threshold_adaptive_spread,
    threshold_query_length_adaptive,
)
from kalmanorix.types import Embedder, Vec
from kalmanorix.uncertainty import KeywordSigma2
from kalmanorix.visualization import (
    plot_embedding_with_uncertainty,
    plot_fusion_weights,
)

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class KeywordEmbedder(Embedder):
    """Toy keyword-sensitive embedder."""

    dim: int
    keywords: Set[str]
    keyword_boost: float
    _base_dir: np.ndarray
    _kw_dir: np.ndarray

    def __call__(self, text: str) -> Vec:
        t = text.lower()

        # Tiny deterministic "noise" so different texts differ slightly.
        noise = np.zeros(self.dim, dtype=np.float64)
        for ch in t[:64]:
            noise[(ord(ch) * 13) % self.dim] += 0.01

        vec = self._base_dir + noise

        if any(kw in t for kw in self.keywords):
            vec = vec + self.keyword_boost * self._kw_dir

        vec = vec / (np.linalg.norm(vec) + 1e-12)
        return vec.astype(np.float64)


def make_keyword_embedder(
    *,
    dim: int,
    seed: int,
    keywords: Set[str],
    keyword_boost: float = 2.5,
) -> KeywordEmbedder:
    """Construct a deterministic keyword-sensitive embedder."""
    rng = np.random.default_rng(seed)

    base_dir = rng.normal(size=(dim,))
    base_dir = base_dir / (np.linalg.norm(base_dir) + 1e-12)

    kw_dir = rng.normal(size=(dim,))
    kw_dir = kw_dir / (np.linalg.norm(kw_dir) + 1e-12)

    return KeywordEmbedder(
        dim=dim,
        keywords=keywords,
        keyword_boost=keyword_boost,
        _base_dir=base_dir.astype(np.float64),
        _kw_dir=kw_dir.astype(np.float64),
    )

In [ ]:
# Create toy specialists with different domains
dim = 32

tech_keywords: Set[str] = {
    "battery",
    "smartphone",
    "cpu",
    "gpu",
    "laptop",
    "android",
    "ios",
    "camera",
    "charger",
    "processor",
}

cook_keywords: Set[str] = {
    "braise",
    "simmer",
    "slow cooker",
    "recipe",
    "garlic",
    "onion",
    "saute",
    "oven",
    "stew",
    "bake",
}

medical_keywords: Set[str] = {
    "symptom",
    "diagnosis",
    "treatment",
    "patient",
    "medicine",
    "hospital",
    "doctor",
    "nurse",
    "prescription",
    "recovery",
}

# Create embedders
tech_embedder = make_keyword_embedder(dim=dim, seed=7, keywords=tech_keywords)
cook_embedder = make_keyword_embedder(dim=dim, seed=11, keywords=cook_keywords)
medical_embedder = make_keyword_embedder(dim=dim, seed=13, keywords=medical_keywords)

# Create SEFs with uncertainty
tech = SEF(
    name="tech",
    embed=tech_embedder,
    sigma2=KeywordSigma2(tech_keywords, in_domain_sigma2=0.2, out_domain_sigma2=2.5),
    meta={"domain": "technology", "keywords": list(tech_keywords)},
)

cook = SEF(
    name="cook",
    embed=cook_embedder,
    sigma2=KeywordSigma2(cook_keywords, in_domain_sigma2=0.2, out_domain_sigma2=2.5),
    meta={"domain": "cooking", "keywords": list(cook_keywords)},
)

medical = SEF(
    name="medical",
    embed=medical_embedder,
    sigma2=KeywordSigma2(medical_keywords, in_domain_sigma2=0.2, out_domain_sigma2=2.5),
    meta={"domain": "medical", "keywords": list(medical_keywords)},
)

# Compute domain centroids for semantic routing
# Use calibration texts from each domain
tech_calibration = [
    "Battery life on this smartphone is excellent",
    "The laptop CPU performs well under load",
    "Camera quality and charger compatibility are important",
    "Android updates improve performance and security",
]

cook_calibration = [
    "Braise the beef and simmer for hours",
    "Slow cooker recipe with garlic and onion",
    "Saute vegetables then bake in the oven",
    "Stew tastes better after slow simmering",
]

medical_calibration = [
    "Patient shows symptoms of common cold",
    "Doctor prescribed medicine for treatment",
    "Hospital recovery requires proper care",
    "Nurse monitored vital signs regularly",
]

tech_with_centroid = tech.with_domain_centroid(tech_calibration)
cook_with_centroid = cook.with_domain_centroid(cook_calibration)
medical_with_centroid = medical.with_domain_centroid(medical_calibration)

village = Village([tech_with_centroid, cook_with_centroid, medical_with_centroid])

print(f"Created {len(village.modules)} specialists with domain centroids:")
for m in village.modules:
    print(f"  - {m.name} (domain: {m.meta['domain']})")

## Semantic Routing with Dynamic Thresholding

Semantic routing selects specialists based on cosine similarity between the query embedding and each specialist's domain centroid. 

Dynamic thresholding adjusts the similarity threshold based on:
- Query characteristics (length, content)
- Similarity distribution across specialists
- Desired selectivity (top-k, fraction of max, etc.)

Available threshold heuristics:
1. `threshold_top_k`: Select top k most similar specialists
2. `threshold_relative_to_max`: Threshold = fraction × max similarity
3. `threshold_adaptive_spread`: Adjust based on similarity spread (tight clusters → higher threshold)
4. `threshold_query_length_adaptive`: Longer queries → lower threshold (more specific)


In [ ]:
# Interactive exploration of routing decisions


def explore_routing(query, threshold_type, param1, param2):
    """Visualize routing decisions for given query and threshold parameters."""

    # Create a fast embedder for query encoding (using tech embedder as proxy)
    fast_embedder = tech_embedder

    # Compute query embedding
    query_vec = fast_embedder(query)

    # Compute similarities to domain centroids
    similarities = []
    for m in village.modules:
        if m.domain_centroid is not None:
            sim = float(np.dot(query_vec, m.domain_centroid))
            similarities.append(sim)
        else:
            similarities.append(0.0)

    # Compute threshold based on selected heuristic
    if threshold_type == "top_k":
        threshold = threshold_top_k(query, query_vec, similarities, k=int(param1))
    elif threshold_type == "relative_to_max":
        threshold = threshold_relative_to_max(
            query, query_vec, similarities, fraction=param1, min_threshold=param2
        )
    elif threshold_type == "adaptive_spread":
        threshold = threshold_adaptive_spread(
            query, query_vec, similarities, spread_factor=param1, min_threshold=param2
        )
    elif threshold_type == "query_length":
        threshold = threshold_query_length_adaptive(
            query, query_vec, similarities, length_factor=param1, base_threshold=param2
        )
    else:
        threshold = 0.5  # default

    # Select modules based on threshold
    selected = []
    for i, m in enumerate(village.modules):
        if threshold < 0 or similarities[i] >= threshold:
            selected.append(m)

    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Plot 1: Similarity bars
    ax1 = axes[0, 0]
    names = [m.name for m in village.modules]
    colors = [
        "lightgreen" if sim >= threshold else "lightcoral" for sim in similarities
    ]
    bars = ax1.bar(range(len(names)), similarities, color=colors, alpha=0.7)
    ax1.axhline(
        y=threshold, color="red", linestyle="--", label=f"Threshold: {threshold:.3f}"
    )
    ax1.set_xlabel("Specialist")
    ax1.set_ylabel("Cosine Similarity")
    ax1.set_title("Similarity to Domain Centroids")
    ax1.set_xticks(range(len(names)))
    ax1.set_xticklabels(names, rotation=45, ha="right")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Add value labels
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.02,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    # Plot 2: Selected modules
    ax2 = axes[0, 1]
    selected_names = [m.name for m in selected]
    if selected_names:
        ax2.bar(
            range(len(selected_names)),
            [1] * len(selected_names),
            color="lightgreen",
            alpha=0.7,
        )
        ax2.set_xlabel("Selected Specialist")
        ax2.set_title(f"Selected Modules ({len(selected)}/{len(village.modules)})")
        ax2.set_xticks(range(len(selected_names)))
        ax2.set_xticklabels(selected_names, rotation=45, ha="right")
        ax2.set_ylim(0, 1.5)
    else:
        ax2.text(
            0.5,
            0.5,
            "No modules selected",
            ha="center",
            va="center",
            transform=ax2.transAxes,
            fontsize=12,
        )
        ax2.set_title("No Modules Selected")
    ax2.grid(True, alpha=0.3)

    # Plot 3: Query analysis
    ax3 = axes[1, 0]
    keywords_found = {}
    for m in village.modules:
        if "keywords" in m.meta:
            kw_list = m.meta["keywords"]
            found = sum(1 for kw in kw_list if kw.lower() in query.lower())
            keywords_found[m.name] = found

    if keywords_found:
        ax3.bar(
            range(len(keywords_found)),
            list(keywords_found.values()),
            color="lightblue",
            alpha=0.7,
        )
        ax3.set_xlabel("Specialist")
        ax3.set_ylabel("Keyword Matches")
        ax3.set_title("Keyword Matches in Query")
        ax3.set_xticks(range(len(keywords_found)))
        ax3.set_xticklabels(list(keywords_found.keys()), rotation=45, ha="right")
        # Add value labels
        for i, v in enumerate(keywords_found.values()):
            ax3.text(i, v + 0.1, str(v), ha="center", va="bottom")
    ax3.grid(True, alpha=0.3)

    # Plot 4: Threshold parameters
    ax4 = axes[1, 1]
    param_text = f"Threshold type: {threshold_type}\n"
    param_text += f"Param1: {param1:.3f}\n"
    param_text += f"Param2: {param2:.3f}\n"
    param_text += f"Query length: {len(query)} chars\n"
    param_text += f"Max similarity: {max(similarities):.3f}\n"
    param_text += f"Similarity std: {np.std(similarities):.3f}"

    ax4.text(
        0.1,
        0.5,
        param_text,
        transform=ax4.transAxes,
        fontsize=11,
        verticalalignment="center",
        bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
    )
    ax4.set_title("Parameters & Statistics")
    ax4.axis("off")

    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"Query: '{query}'")
    print(f"Threshold: {threshold:.3f} ({threshold_type})")
    print(f"Selected: {[m.name for m in selected]}")


# Create interactive widgets
query_widget = widgets.Textarea(
    value="This smartphone battery lasts longer than a slow cooker braise but patient recovery is important.",
    description="Query:",
    layout=widgets.Layout(width="80%", height="80px"),
)

threshold_type_widget = widgets.Dropdown(
    options=["top_k", "relative_to_max", "adaptive_spread", "query_length"],
    value="relative_to_max",
    description="Threshold type:",
)

param1_widget = widgets.FloatSlider(
    value=0.8,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Param1:",
    continuous_update=False,
)

param2_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Param2:",
    continuous_update=False,
)

# Connect widgets to function
widgets.interactive(
    explore_routing,
    query=query_widget,
    threshold_type=threshold_type_widget,
    param1=param1_widget,
    param2=param2_widget,
)

## Fusion Strategies Comparison

Once specialists are selected, Kalmanorix can fuse their embeddings using different strategies:

1. **MeanFuser**: Uniform averaging (baseline)
2. **KalmanorixFuser**: Precision-weighted fusion using diagonal covariance
3. **LearnedGateFuser**: Learned gating based on text features

The KalmanorixFuser uses each specialist's uncertainty estimate (sigma²) to weight their contribution: lower uncertainty → higher weight.


In [ ]:
# Interactive fusion visualization


def compare_fusion(query, fusion_strategy, use_semantic_routing, threshold):
    """Compare fusion strategies for a given query."""

    # Create router based on selection
    if use_semantic_routing:
        # Use semantic routing with fixed threshold
        scout = ScoutRouter(
            mode="semantic",
            fast_embedder=tech_embedder,  # proxy fast embedder
            similarity_threshold=threshold,
        )
    else:
        # Use all specialists (fusion mode)
        scout = ScoutRouter(mode="all")

    # Create fuser based on selection
    if fusion_strategy == "mean":
        fuser = MeanFuser()
    elif fusion_strategy == "kalman":
        fuser = KalmanorixFuser()
    elif fusion_strategy == "learned_gate":
        # Train a simple learned gate for demo
        gate_fuser = LearnedGateFuser(
            module_a="tech",
            module_b="cook",
            n_features=128,
            lr=0.6,
            l2=1e-3,
            steps=400,
        )
        # Tiny training data
        train_texts = [
            "Battery life is excellent on this smartphone",
            "The laptop CPU throttles under load",
            "Camera quality and charger compatibility",
            "Android update improved performance",
            "Braise the beef and simmer for hours",
            "Slow cooker recipe with garlic and onion",
            "Saute the vegetables then bake in the oven",
            "Stew tastes better after simmering",
        ]
        train_y = [1, 1, 1, 1, 0, 0, 0, 0]  # 1=tech, 0=cooking
        gate_fuser.fit(train_texts, train_y)
        fuser = gate_fuser
    else:
        fuser = MeanFuser()

    # Create panoramix and brew potion
    panoramix = Panoramix(fuser=fuser)
    potion = panoramix.brew(query, village=village, scout=scout)

    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot 1: Fusion weights
    ax1 = axes[0]
    weights_list = [(name, weight) for name, weight in potion.weights.items()]
    plot_fusion_weights(weights_list, ax=ax1, show=False)
    ax1.set_title(f"{fusion_strategy.title()} Fusion Weights")

    # Plot 2: Embedding with uncertainty (first specialist)
    ax2 = axes[1]
    if village.modules:
        first_module = village.modules[0]
        emb = first_module.embed(query)
        sigma2 = first_module.sigma2_for(query)
        # Create diagonal covariance vector
        cov = np.full(emb.shape, sigma2, dtype=np.float64)
        plot_embedding_with_uncertainty(
            emb, cov, title=f"{first_module.name} Embedding ±1σ", ax=ax2, show=False
        )

    # Plot 3: Routing information
    ax3 = axes[2]
    info_text = f"Strategy: {fusion_strategy}\n"
    info_text += f"Routing: {'semantic' if use_semantic_routing else 'all'}\n"
    if use_semantic_routing:
        info_text += f"Threshold: {threshold}\n"
    info_text += f"Selected: {list(potion.weights.keys())}\n"
    info_text += f"Total weight: {sum(potion.weights.values()):.3f}\n"

    if potion.meta and "variance" in potion.meta:
        info_text += f"Fused variance: {potion.meta['variance']:.4f}"

    ax3.text(
        0.1,
        0.5,
        info_text,
        transform=ax3.transAxes,
        fontsize=11,
        verticalalignment="center",
        bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
    )
    ax3.set_title("Fusion Metadata")
    ax3.axis("off")

    plt.tight_layout()
    plt.show()

    # Print detailed weights
    print(f"\nDetailed weights (query: '{query[:50]}...'):")
    for name, weight in potion.weights.items():
        print(f"  {name}: {weight:.4f}")


# Create widgets for fusion comparison
fusion_query_widget = widgets.Textarea(
    value="Patient needs smartphone for telemedicine appointments and recipe suggestions.",
    description="Query:",
    layout=widgets.Layout(width="80%", height="80px"),
)

fusion_strategy_widget = widgets.Dropdown(
    options=["mean", "kalman", "learned_gate"],
    value="kalman",
    description="Fusion strategy:",
)

routing_widget = widgets.Checkbox(value=True, description="Use semantic routing")

threshold_widget = widgets.FloatSlider(
    value=0.6, min=0.0, max=1.0, step=0.05, description="Threshold:", disabled=False
)

# Connect widgets
widgets.interactive(
    compare_fusion,
    query=fusion_query_widget,
    fusion_strategy=fusion_strategy_widget,
    use_semantic_routing=routing_widget,
    threshold=threshold_widget,
)

## Conclusion and Next Steps

This interactive demo has shown:

1. **Semantic Routing**: How queries can be routed to relevant specialists based on domain similarity
2. **Dynamic Thresholding**: How threshold heuristics adapt to query characteristics
3. **Kalman Fusion**: How uncertainty estimates weight specialist contributions
4. **Visualization**: Real-time exploration of routing decisions and fusion weights

### Try These Examples:

- Pure tech query: "Smartphone battery life and CPU performance"
- Pure cooking query: "Slow cooker recipe with garlic and onion"
- Mixed query: "Patient using smartphone for telemedicine while cooking"
- Long specific query: "Detailed diagnosis of smartphone battery drain issues while preparing complex recipes"

### Advanced Exploration:

1. Adjust threshold parameters to see how selectivity changes
2. Compare fusion strategies on mixed-domain queries
3. Observe how query length affects threshold in adaptive strategies
4. Note how keyword matches correlate with similarity scores

### Production Usage:

For production deployments:
- Replace toy embedders with real models (Sentence Transformers, OpenAI, etc.)
- Use `OpenAIEmbedder` or other adapters from `embedder_adapters.py`
- Compute domain centroids on representative domain texts
- Choose threshold heuristic based on your use case
- Monitor routing accuracy and fusion performance
